# 01 — Preparation

Half the battle in matching is getting your data into a shape matcher can use—and knowing how you'll measure success. Here we walk through schema and identifiers, **schema standardization** (same column names on both sides), nulls, value-level standardization, derived columns, and ground truth. We use the entity resolution sample data so you can run every step yourself.

**Back:** [Preamble & TOC](00_preamble_and_toc.ipynb) · **Next:** [02 Exact Matching](02_exact_matching.ipynb)

## 1. Load the data (messy on purpose)

We start with data as it might arrive: **column name mismatches** (e.g. right has `email_address` and `postal_code` instead of `email` and `zip_code`), **nulls**, and **messy values** (spaces, mixed case). Matcher needs the same column names on both sides and clean values for matching—so we'll fix all of this step by step.

**Data:** 500 left and 500 right rows, **30 known true pairs** (exact name, email-adds-value, fuzzy name). Load and inspect: columns, nulls, and a peek at the mess.

In [1]:
import sys
from pathlib import Path
import polars as pl

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_entity_resolution

left, right, ground_truth = load_entity_resolution()

Inspect **schema**: do both tables use the same column names and shapes?

In [2]:
print("Left columns:", left.columns)
print("Right columns:", right.columns)
print("Same names?", set(left.columns) == set(right.columns))
print("Left shape:", left.shape, " Right shape:", right.shape)

Left columns: ['id', 'email', 'first_name', 'last_name', 'address', 'city', 'state', 'zip_code']
Right columns: ['id', 'email_address', 'first_name', 'last_name', 'address', 'city', 'state', 'postal_code']
Same names? False
Left shape: (500, 8)  Right shape: (500, 8)


Check **nulls** in key columns (different names per side until we standardize).

In [3]:
null_cols_left = ["email", "first_name", "zip_code"]
null_cols_right = ["email_address", "first_name", "postal_code"]
print("Nulls (left):", {c: left[c].null_count() for c in null_cols_left})
print("Nulls (right):", {c: right[c].null_count() for c in null_cols_right})

Nulls (left): {'email': 5, 'first_name': 0, 'zip_code': 3}
Nulls (right): {'email_address': 0, 'first_name': 4, 'postal_code': 3}


Peek at **messy values** (e.g. extra spaces, mixed case) before we clean them.

In [4]:
print("Sample messy values (left email, right email_address):")
display(
    left.filter(pl.col("email").str.contains("  ").fill_null(False))
    .select(["email"])
    .head(3)
)
display(
    right.filter(pl.col("email_address").str.contains("  ").fill_null(False))
    .select(["email_address"])
    .head(3)
)

Sample messy values (left email, right email_address):


email
str
""" michael.taylor.5@ACME.ORG """
""" linda.moore.6@ACME.ORG """
""" david.jackson.7@ACME.ORG """


email_address
str
""" sharon.morris.8@GLOBEX.COM """
""" timothy.murphy.9@GLOBEX.COM …"
""" cynthia.cook.10@GLOBEX.COM """


Next we standardize schema (rename right to match left), then nulls, then value-level cleaning.

## 2. Schema standardization

In real data, the two sources often use different column names (e.g. `email` vs `email_address`, or `customer_id` vs `id`). Matcher joins on **column names**, so any field you match or block on must have the same name on both sides. Standardize by renaming columns on one or both tables to a common schema, and optionally select a canonical column order.

Rename right to match left (`email_address` → `email`, `postal_code` → `zip_code`), then align both to the same column order.

In [5]:
# Rename right to match left, then align both to the same column order
right = right.rename({"email_address": "email", "postal_code": "zip_code"})
expected_schema = [
    "id", "email", "first_name", "last_name",
    "address", "city", "state", "zip_code",
]
left = left.select(expected_schema)
right = right.select(expected_schema)
print("Schema standardized. Both tables:", list(left.columns))

Schema standardized. Both tables: ['id', 'email', 'first_name', 'last_name', 'address', 'city', 'state', 'zip_code']


## 3. Nulls and missing values

Nulls affect what gets compared. Matcher doesn't guess: for **exact** rules, a null in a join field means that row won't match (inner join). For **fuzzy**, rows with null in the fuzzy field are skipped. For **blocking**, nulls in the blocking key end up in one block. So it's worth checking null counts and deciding up front whether to drop, fill, or leave as-is.

Check nulls in the columns we care about.

In [6]:
cols = ["id", "email", "first_name", "last_name", "zip_code"]
for col in cols:
    n_left = left[col].null_count()
    n_right = right[col].null_count()
    print(f"{col}: left nulls={n_left}, right nulls={n_right}")

id: left nulls=0, right nulls=0
email: left nulls=5, right nulls=0
first_name: left nulls=0, right nulls=4
last_name: left nulls=0, right nulls=0
zip_code: left nulls=3, right nulls=3


This sample has a few nulls in email (left), first_name (right), and zip_code (both) so you see real counts. Decide how to handle them before matching—e.g. drop rows, fill with a sentinel, or leave as-is. Matcher doesn't fill nulls for you.

## 4. Value-level standardization (cleaning)

Our data has **messy values**: some emails have leading/trailing spaces and mixed case; some name fields have extra spaces. Exact match compares literally, so we need to clean. We'll do it step by step, then encapsulate in a function you can reuse after loading any data.

In [7]:
# Step 1: Clean email — lowercase + strip (keep original, add email_clean)
email_clean = (
    pl.col("email").cast(pl.Utf8).str.to_lowercase().str.strip_chars().alias("email_clean")
)
left = left.with_columns(email_clean)
right = right.with_columns(email_clean)
print("Before → after (left, messy rows):")
display(
    left.filter(pl.col("email").str.contains("  ").fill_null(False))
    .select(["email", "email_clean"])
    .head(5)
)

Before → after (left, messy rows):


email,email_clean
str,str
""" michael.taylor.5@ACME.ORG ""","""michael.taylor.5@acme.org"""
""" linda.moore.6@ACME.ORG ""","""linda.moore.6@acme.org"""
""" david.jackson.7@ACME.ORG ""","""david.jackson.7@acme.org"""
""" barbara.martin.8@ACME.ORG ""","""barbara.martin.8@acme.org"""
""" william.lee.9@ACME.ORG ""","""william.lee.9@acme.org"""


Below we show a sample of messy rows (email, first_name) for illustration. In Section 5 we add a **full_name** column (first + last, stripped) for fuzzy matching and blocking.

In [8]:
# Show the mess: some emails have spaces and mixed case, some names have extra spaces
print("Left — rows with messy email or first_name (e.g. leading/trailing spaces):")
display(
    left.filter(
        pl.col("email").str.ends_with("  ")
        | pl.col("first_name").str.starts_with("  ")
    )
    .select(["id", "email", "first_name"])
    .head(8)
)
print("Right — sample of messy emails:")
display(
    right.filter(pl.col("email").str.contains("  "))
    .select(["id", "email", "first_name"])
    .head(5)
)

Left — rows with messy email or first_name (e.g. leading/trailing spaces):


id,email,first_name
str,str,str
"""left_46""",""" michael.taylor.5@ACME.ORG ""","""Michael"""
"""left_47""",""" linda.moore.6@ACME.ORG ""","""Linda"""
"""left_48""",""" david.jackson.7@ACME.ORG ""","""David"""
"""left_49""",""" barbara.martin.8@ACME.ORG ""","""Barbara"""
"""left_50""",""" william.lee.9@ACME.ORG ""","""William"""
"""left_51""",""" elizabeth.perez.10@ACME.ORG …","""Elizabeth"""
"""left_52""",""" richard.thompson.11@ACME.ORG…","""Richard"""
"""left_53""",""" susan.white.12@ACME.ORG ""","""Susan"""


Right — sample of messy emails:


id,email,first_name
str,str,str
"""right_49""",""" sharon.morris.8@GLOBEX.COM ""","""Sharon"""
"""right_50""",""" timothy.murphy.9@GLOBEX.COM …","""Timothy"""
"""right_51""",""" cynthia.cook.10@GLOBEX.COM ""","""Cynthia"""
"""right_52""",""" jason.rogers.11@GLOBEX.COM ""","""Jason"""
"""right_53""",""" kathleen.morgan.12@GLOBEX.CO…","""Kathleen"""


## 5. Feature engineering for matching and blocking

Fuzzy matching in matcher works on a **single** string column. For names, combining first + last into `full_name` gives you one field to score. Same idea for blocking: if you don't have a natural key, derive one (e.g. first letter of surname) the same way on both sides.

In [9]:
full_name_expr = (
    (pl.col("first_name").fill_null("") + " " + pl.col("last_name").fill_null(""))
    .str.strip_chars()
    .alias("full_name")
)
left = left.with_columns(full_name_expr)
right = right.with_columns(full_name_expr)
print(left.select(["first_name", "last_name", "full_name"]).head(3))

shape: (3, 3)
┌────────────┬───────────┬─────────────┐
│ first_name ┆ last_name ┆ full_name   │
│ ---        ┆ ---       ┆ ---         │
│ str        ┆ str       ┆ str         │
╞════════════╪═══════════╪═════════════╡
│ Alice      ┆ Smith     ┆ Alice Smith │
│ Bob        ┆ Jones     ┆ Bob Jones   │
│ Carol      ┆ White     ┆ Carol White │
└────────────┴───────────┴─────────────┘


We already have `zip_code` for blocking. In other datasets you might derive a weaker blocker (e.g. first letter of surname)—just keep the derivation identical on both tables.

## 6. Encapsulate: standardize once, use everywhere

We've done schema alignment, null checks, email cleaning, and full_name. **Best practice:** put these steps in one function and call it after loading raw data. Then matching notebooks (02 onward) can load **already standardized** data and focus on rules and evaluation. Raw data → clean it → clean data → start matching.

The same steps above are in `standardize_for_matching(left, right)` in `tutorial_data`. From 02 on we use `load_entity_resolution_standardized()` so we get clean data without re-running prep.

In [10]:
from tutorial_data import standardize_for_matching

# Re-run the same pipeline so one entry point; from 02 on we load standardized data directly.
left, right = standardize_for_matching(left, right)

# Proof: same schema and derived columns on both sides
print("Same columns?", set(left.columns) == set(right.columns))
print("Left shape:", left.shape, " Right shape:", right.shape)
print("Derived columns present:", "email_clean" in left.columns and "full_name" in left.columns)
print()
print("Sample (left) — standardized email_clean and full_name:")
display(left.select(["email", "email_clean", "full_name"]).head(3))

Same columns? True
Left shape: (500, 10)  Right shape: (500, 10)
Derived columns present: True

Sample (left) — standardized email_clean and full_name:


email,email_clean,full_name
str,str,str
"""alice.smith@acme.org""","""alice.smith@acme.org""","""Alice Smith"""
"""bob.jones@acme.org""","""bob.jones@acme.org""","""Bob Jones"""
"""carol.white@acme.org""","""carol.white@acme.org""","""Carol White"""


## 7. Ground truth for evaluation

To know whether your algorithm is doing well, you need a set of **known true pairs**: a DataFrame with `left_id` and `right_id`. You might get that from a golden set, from business rules (e.g. "same email ⇒ same person"), or from manual review. We loaded `ground_truth` with the entity_resolution data (30 known pairs).

In [11]:
print("Ground truth (first 5 rows):")
print(ground_truth.head(5))

Ground truth (first 5 rows):
shape: (5, 2)
┌─────────┬──────────┐
│ left_id ┆ right_id │
│ ---     ┆ ---      │
│ str     ┆ str      │
╞═════════╪══════════╡
│ left_1  ┆ right_1  │
│ left_2  ┆ right_2  │
│ left_3  ┆ right_3  │
│ left_4  ┆ right_4  │
│ left_5  ┆ right_5  │
└─────────┴──────────┘


When you run matching in the next notebooks, you'll pass this (or your own) ground truth into `results.evaluate(...)` so you can compare precision, recall, and F1 across different rules and settings.

## 8. Before you move on

You're ready for [02 Exact Matching](02_exact_matching.ipynb) when: schema is standardized (same names on both sides), IDs and match/blocking columns are aligned, you've decided how to handle nulls, you've added any value-level standardization or derived columns you need, and you have ground truth to evaluate against. In this sample we've done all of that.

**Next:** [02 Exact Matching](02_exact_matching.ipynb) — single rule, cascading rules, and evaluation.